# Lenta Colab Runs Compare

Reads artifacts saved by the experiment notebooks from `MyDrive/lenta_colab_runs`,
builds a compact comparison table, and prints the next iteration targets.

In [ ]:
import json
from pathlib import Path

import pandas as pd

from google.colab import drive
drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/lenta_colab_runs")
print(ROOT)

In [ ]:
def read_json(path):
    if not path.exists():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


rows = []
for run_dir in sorted(ROOT.glob("*")):
    artifacts = run_dir / "artifacts"
    if not artifacts.exists():
        continue
    summary = read_json(artifacts / "run_summary.json")
    yolo = read_json(artifacts / "kaggle_metrics" / "detection_recall_yolo_only.json")
    hybrid = read_json(artifacts / "kaggle_metrics" / "detection_recall_hybrid.json")
    metrics = read_json(artifacts / "eval_public_fast" / "metrics.json")
    row = {
        "run": run_dir.name,
        "experiment": summary.get("experiment_name", ""),
        "elapsed_sec": summary.get("elapsed_sec", ""),
        "yolo_recall": yolo.get("recall_at_iou_035", ""),
        "hybrid_recall": hybrid.get("recall_at_iou_035", ""),
    }
    total_gt = total_pred = total_matched = total_good = 0
    for video, m in metrics.items():
        total_gt += int(m.get("gt_rows", 0))
        total_pred += int(m.get("pred_rows", 0))
        total_matched += int(m.get("matched_rows", 0))
        total_good += int(m.get("good_rows_at_80", 0))
        row[f"{video}_pred"] = m.get("pred_rows", "")
        row[f"{video}_matched"] = m.get("matched_rows", "")
        row[f"{video}_good80"] = m.get("good_rows_at_80", "")
    row.update({
        "gt_total": total_gt,
        "pred_total": total_pred,
        "matched_total": total_matched,
        "good80_total": total_good,
    })
    rows.append(row)

df = pd.DataFrame(rows)
if df.empty:
    print("No runs found. Run A/B/C notebooks first.")
else:
    display(df.sort_values(["good80_total", "matched_total"], ascending=False))

In [ ]:
# Field-fill and duplicate/no-evidence diagnostics.
diag_rows = []
for run_dir in sorted(ROOT.glob("*")):
    err_dir = run_dir / "artifacts" / "error_analysis"
    for path in sorted(err_dir.glob("*_error_analysis.json")):
        rep = read_json(path)
        fill = rep.get("field_fill", {})
        diag_rows.append({
            "run": run_dir.name,
            "video": path.name.replace("_error_analysis.json", ""),
            "gt": rep.get("gt_rows"),
            "pred": rep.get("pred_rows"),
            "matched": rep.get("matched_rows"),
            "no_evidence": len(rep.get("no_semantic_evidence_pred", [])),
            "dup_clusters": len(rep.get("duplicate_clusters", [])),
            "product_name": fill.get("product_name", 0),
            "price_default": fill.get("price_default", 0),
            "price_card": fill.get("price_card", 0),
            "barcode": fill.get("barcode", 0),
            "qr_code_barcode": fill.get("qr_code_barcode", 0),
            "id_sku": fill.get("id_sku", 0),
        })
diag = pd.DataFrame(diag_rows)
if diag.empty:
    print("No error-analysis reports found.")
else:
    display(diag)

## Harsh decision rule

- If detector recall rises but `good80_total` stays zero, stop training detector and fix QR/OCR/parser/fusion.
- If `pred_total` is above `1.35 * gt_total`, fallback/dedupe is still broken.
- If `qr_code_barcode` remains near zero, QR cascade/layout crop is the next blocker.
- If prices are still empty, zonal OCR/parser is still the first failure path.